In [1]:
import gc
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

In [2]:
def load_and_clean_supplementary():
    # 1. Customers
    df_customers = pd.read_csv('customers.csv')
    df_customers['region'] = (
        df_customers['region']
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r'[\s_-]+', '-', regex=True)
    )
    df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'], errors='coerce')
    df_customers['record_updated_at'] = pd.to_datetime(df_customers['record_updated_at'], errors='coerce')
    
    # Handle duplicates if present in customer records (keep most recent)
    df_customers = df_customers.sort_values('record_updated_at').drop_duplicates(subset=['customer_id'], keep='last')

    # 2. Products
    df_products = pd.read_csv('products.csv')
    df_products['launch_date'] = pd.to_datetime(df_products['launch_date'], errors='coerce')
    # Physical and volumetric features
    df_products['vol_cm3'] = df_products['length_cm'] * df_products['width_cm'] * df_products['height_cm']
    df_products['density'] = df_products['weight_kg'] / (df_products['vol_cm3'] + 1e-4)

    # 3. Stores
    df_stores = pd.read_csv('stores.csv')
    df_stores['opened_date'] = pd.to_datetime(df_stores['opened_date'], errors='coerce')
    df_stores['region'] = (
        df_stores['region']
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r'[\s_-]+', '-', regex=True)
    )

    # 4. Calendar
    df_calendar = pd.read_csv('calendar.csv')
    df_calendar['week_start'] = pd.to_datetime(df_calendar['week_start'])

    return df_customers, df_products, df_stores, df_calendar

df_customers, df_products, df_stores, df_calendar = load_and_clean_supplementary()
print("Supplementary tables loaded and cleaned.")

Supplementary tables loaded and cleaned.


In [3]:
def preprocess_transactions(df, df_customers, df_products, df_stores, df_calendar):
    df['order_ts'] = pd.to_datetime(df['order_ts'])
    
    # Align order_ts to Monday of that week for calendar merge
    df['week_start'] = df['order_ts'].dt.to_period('W-SUN').apply(lambda r: r.start_time)

    # Merge supplementary tables
    df = df.merge(df_customers, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(df_products, on='product_id', how='left', suffixes=('', '_prod'))
    df = df.merge(df_stores, on='store_id', how='left', suffixes=('', '_store'))
    df = df.merge(df_calendar, on='week_start', how='left')

    # 1. Temporal features
    df['order_year'] = df['order_ts'].dt.year
    df['order_month'] = df['order_ts'].dt.month
    df['order_day'] = df['order_ts'].dt.day
    df['order_dayofweek'] = df['order_ts'].dt.dayofweek
    df['order_hour'] = df['order_ts'].dt.hour
    df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(np.int8)

    # Lifecycle & tenure intervals (in days)
    df['days_since_signup'] = (df['order_ts'] - df['signup_date']).dt.days.clip(lower=0)
    df['days_since_prod_launch'] = (df['order_ts'] - df['launch_date']).dt.days.clip(lower=0)
    df['days_since_store_open'] = (df['order_ts'] - df['opened_date']).dt.days.clip(lower=0)

    # 2. Pricing, Discount & Order Line Ratios
    df['calculated_line_total'] = df['unit_price'] * df['quantity'] * (1 - df['discount'].fillna(0))
    df['price_ratio_list'] = df['unit_price'] / (df['list_price'] + 1e-4)
    df['discount_per_unit'] = (df['list_price'] - df['unit_price']).clip(lower=0)
    df['freight_share'] = df['freight_cost'] / (df['order_value'] + 1e-4)
    df['freight_per_item'] = df['freight_cost'] / (df['quantity'] + 1e-4)
    df['line_position_ratio'] = df['line_no'] / (df['items_in_order'] + 1e-4)
    df['has_coupon'] = df['coupon_code'].notnull().astype(np.int8)

    # 3. Macro & Environmental Interaction
    df['cost_pressure_index'] = df['fuel_price'] * (df['fuel_surcharge'].fillna(0) + 1.0)

    # Drop unneeded identifier & raw timestamp objects to save memory
    drop_cols = [
        'order_ts', 'signup_date', 'launch_date', 'opened_date', 'record_updated_at',
        'week_start', 'coupon_code'
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    
    return df

print("Preprocess function ready.")

Preprocess function ready.


In [4]:
def add_order_level_aggregations(df):
    """Computes context for the basket (e.g. basket size, diversity)."""
    order_grp = df.groupby('order_id').agg(
        basket_total_items=('quantity', 'sum'),
        basket_max_price=('unit_price', 'max'),
        basket_min_price=('unit_price', 'min'),
        basket_mean_discount=('discount', 'mean'),
        basket_distinct_products=('product_id', 'nunique')
    ).reset_index()

    df = df.merge(order_grp, on='order_id', how='left')
    df['item_price_relative_to_max'] = df['unit_price'] / (df['basket_max_price'] + 1e-4)
    return df

In [5]:
def compute_smoothed_target_encoding(train_df, test_df, cat_cols, target='returned', m=50):
    """
    Computes smoothed Bayesian out-of-fold target encoding for high-cardinality features 
    to prevent target leakage.
    Formula: S_i = (count_i * mean_i + m * prior) / (count_i + m)
    """
    global_mean = train_df[target].mean()
    
    for col in cat_cols:
        enc_col_name = f'{col}_return_rate'
        
        # Calculate full stats on train for test mapping
        stats = train_df.groupby(col)[target].agg(['count', 'mean'])
        smoothed = (stats['count'] * stats['mean'] + m * global_mean) / (stats['count'] + m)
        mapping = smoothed.to_dict()
        test_df[enc_col_name] = test_df[col].map(mapping).fillna(global_mean).astype(np.float32)

        # OOF calculation for train
        train_df[enc_col_name] = np.nan
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        for tr_idx, val_idx in skf.split(train_df, train_df[target]):
            fold_stats = train_df.iloc[tr_idx].groupby(col)[target].agg(['count', 'mean'])
            fold_smoothed = (fold_stats['count'] * fold_stats['mean'] + m * global_mean) / (fold_stats['count'] + m)
            train_df.iloc[val_idx, train_df.columns.get_loc(enc_col_name)] = (
                train_df.iloc[val_idx][col].map(fold_smoothed.to_dict()).fillna(global_mean)
            )
        train_df[enc_col_name] = train_df[enc_col_name].astype(np.float32)
        
    return train_df, test_df

In [6]:
print("Loading train.csv and test.csv...")
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Feature pipeline execution
df_train = preprocess_transactions(df_train, df_customers, df_products, df_stores, df_calendar)
df_test = preprocess_transactions(df_test, df_customers, df_products, df_stores, df_calendar)

df_train = add_order_level_aggregations(df_train)
df_test = add_order_level_aggregations(df_test)

# High-signal entities for smoothed target encoding
target_enc_cols = ['category', 'promo_channel_code', 'store_id', 'loyalty_tier', 'packaging_type']
df_train, df_test = compute_smoothed_target_encoding(df_train, df_test, target_enc_cols)

# Ensure categorical data types for LightGBM native handling
categorical_cols = [
    'channel', 'payment_method', 'ship_mode', 'promo_channel_code',
    'warehouse_zone', 'picker_shift', 'device_type', 'tz_hint',
    'region', 'customer_type', 'age_band', 'preferred_contact', 'loyalty_tier',
    'category', 'packaging_type', 'supplier_code', 'color',
    'channel_store', 'region_store', 'format'
]

for col in categorical_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype('category')
        df_test[col] = df_test[col].astype('category')

gc.collect()
print("Data processing and feature matrix construction completed.")

Loading train.csv and test.csv...
Data processing and feature matrix construction completed.


In [7]:
# Identify feature list (exclude IDs and label)
ignore_cols = ['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id', 'returned']
features = [c for c in df_train.columns if c not in ignore_cols]
target = 'returned'

print(f"Number of training features: {len(features)}")

# Time-based validation split:
# Since test is Months 25-30, use the last 4 months of train as local validation set
# (Train covers 2023-01 to 2024-12, so train on <= 2024-08, validate on > 2024-08)
is_val = (df_train['order_year'] == 2024) & (df_train['order_month'] >= 9)

X_tr, y_tr = df_train.loc[~is_val, features], df_train.loc[~is_val, target]
X_val, y_val = df_train.loc[is_val, features], df_train.loc[is_val, target]

print(f"Train set: {X_tr.shape[0]} rows | Validation set: {X_val.shape[0]} rows")

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'min_child_samples': 50,
    'n_jobs': -1,
    'random_state': 42,
    'verbose': -1
}

train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

evals_result = {}
model = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=3000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=80),
        lgb.log_evaluation(period=100),
        lgb.record_evaluation(evals_result)
    ]
)

val_preds = model.predict(X_val, num_iteration=model.best_iteration)
val_auc = roc_auc_score(y_val, val_preds)
print(f"\n==========================================")
print(f"Hold-out Time Validation ROC-AUC: {val_auc:.5f}")
print(f"==========================================")

Number of training features: 77
Train set: 1165899 rows | Validation set: 234106 rows
Training until validation scores don't improve for 80 rounds
[100]	train's auc: 0.765976	val's auc: 0.644736
[200]	train's auc: 0.78536	val's auc: 0.654334
[300]	train's auc: 0.797381	val's auc: 0.656107
[400]	train's auc: 0.805658	val's auc: 0.65863
[500]	train's auc: 0.81118	val's auc: 0.659427
[600]	train's auc: 0.815906	val's auc: 0.660356
[700]	train's auc: 0.820161	val's auc: 0.660667
[800]	train's auc: 0.824284	val's auc: 0.661352
[900]	train's auc: 0.828049	val's auc: 0.661285
Early stopping, best iteration is:
[822]	train's auc: 0.825075	val's auc: 0.661456

Hold-out Time Validation ROC-AUC: 0.66146


In [8]:
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("Top 20 Features by Information Gain:")
print(importance_df.head(20).to_string(index=False))

Top 20 Features by Information Gain:
                       feature   importance
                 supplier_code 1.116413e+06
promo_channel_code_return_rate 6.331903e+05
            promo_channel_code 1.224497e+05
          store_id_return_rate 8.262873e+04
                        region 4.225135e+04
                 customer_type 4.130076e+04
                items_in_order 2.460794e+04
                     weight_kg 2.285280e+04
                  freight_cost 2.260631e+04
                           cpi 2.218870e+04
                      category 2.059218e+04
                  region_store 2.009835e+04
                 freight_share 2.008123e+04
                      discount 1.938071e+04
        days_since_prod_launch 1.687424e+04
             days_since_signup 1.272213e+04
                    unit_price 1.264475e+04
                        format 1.261995e+04
                      quantity 1.191441e+04
      basket_distinct_products 1.098031e+04


In [9]:
print("Retraining LightGBM on 100% of historical data for final submission...")

# Train using optimal iterations from validation
optimal_rounds = int(model.best_iteration * 1.15)
full_train_data = lgb.Dataset(df_train[features], label=df_train[target])

final_model = lgb.train(
    lgb_params,
    full_train_data,
    num_boost_round=optimal_rounds,
    callbacks=[lgb.log_evaluation(period=200)]
)

# Predict test set probabilities
test_preds = final_model.predict(df_test[features])

# Verify format with sample submission
sub = pd.DataFrame({
    'order_line_id': df_test['order_line_id'],
    'returned': test_preds
})

sub.to_csv('submission_lgb.csv', index=False)
print(f"Submission saved: {sub.shape} rows.")
print(sub.head())

Retraining LightGBM on 100% of historical data for final submission...
Submission saved: (479995, 2) rows.
     order_line_id  returned
0  L85d608999fa1ee  0.166018
1  L86272cb9543018  0.171004
2  L66b5669bdace5b  0.093217
3  L2c5886124612de  0.137891
4  L56fe7aa5ac5dd8  0.115429
